## Cell 1: Set up

In [ ]:
import boto3, io, json, os
import numpy as np
import pandas as pd
import sagemaker
from sagemaker import RandomCutForest
from sagemaker.serverless import ServerlessInferenceConfig

BUCKET = "psb-lab-<DEPLOYMENT-ID>"      # <-- edit this
RAW_PREFIX = "raw/"
REGION = "ap-southeast-1"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
s3 = boto3.client("s3", region_name=REGION)

print("SageMaker role :", role)
print("Default bucket :", session.default_bucket())
print("Lab bucket     :", BUCKET)

## Cell 2: Load the raw telemetry

In [ ]:
def load_raw(bucket, prefix, limit=20000):
    """Read the IoT rule's JSON objects into a DataFrame."""
    paginator = s3.get_paginator("list_objects_v2")
    records, seen = [], 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(".json"):
                continue
            body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
            try:
                records.append(json.loads(body))
            except json.JSONDecodeError:
                continue
            seen += 1
            if seen >= limit:
                break
        if seen >= limit:
            break
    return pd.DataFrame(records)

df = load_raw(BUCKET, RAW_PREFIX)
print(f"Loaded {len(df):,} readings")
df.head()

## Cell 3: Prepare the feature matrix

In [ ]:
FEATURES = ["temperature_c", "humidity_pct", "co2_ppm", "vibration_g"]

df["ts"] = pd.to_numeric(df["ts"], errors="coerce")
df = df.dropna(subset=["ts"] + FEATURES).sort_values("ts").reset_index(drop=True)

# Keep the injected_anomaly flag ASIDE for evaluation only.
# It is never used for training - Random Cut Forest is unsupervised.
truth = df["injected_anomaly"].fillna(False).astype(bool).values \
    if "injected_anomaly" in df.columns else None

X = df[FEATURES].astype("float32").values
print("Feature matrix:", X.shape)
print(df[FEATURES].describe().round(2))

## Cell 4: Train the model

In [ ]:
rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    data_location=f"s3://{BUCKET}/model/rcf-train/",
    output_path=f"s3://{BUCKET}/model/",
    num_samples_per_tree=256,
    num_trees=64,
    base_job_name="psb-rcf",
    sagemaker_session=session,
)

rcf.fit(rcf.record_set(X))
print("Training job:", rcf.latest_training_job.name)

## Cell 5: Deploy to a serverless endpoint

In [ ]:
predictor = rcf.deploy(
    endpoint_name="psb-anomaly-endpoint",
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=2048,
        max_concurrency=5,
    ),
)
print("Endpoint:", predictor.endpoint_name)

## Cell 6: Score the training data and evaluate

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

def score(matrix, batch=500):
    out = []
    for i in range(0, len(matrix), batch):
        chunk = matrix[i:i + batch]
        result = predictor.predict(chunk)
        out.extend(r["score"] for r in result["scores"])
    return np.array(out)

scores = score(X)
df["anomaly_score"] = scores

cutoff = np.percentile(scores, 98.0)     # flag the top 2%
df["is_anomaly"] = df["anomaly_score"] > cutoff

print(f"Score range     : {scores.min():.3f} .. {scores.max():.3f}")
print(f"Mean score      : {scores.mean():.3f}")
print(f"98th percentile : {cutoff:.3f}")
print(f"Flagged         : {df['is_anomaly'].sum():,} of {len(df):,}")

## Cell 7: Check the model against the injected anomalies

In [ ]:
if truth is not None and truth.any():
    tp = int((df["is_anomaly"] &  truth).sum())
    fp = int((df["is_anomaly"] & ~truth).sum())
    fn = int((~df["is_anomaly"] &  truth).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"True positives : {tp}")
    print(f"False positives: {fp}")
    print(f"False negatives: {fn}")
    print(f"Precision      : {precision:.3f}")
    print(f"Recall         : {recall:.3f}")
    print(f"F1             : {f1:.3f}")
else:
    print("No ground-truth column - your IoT rule correctly dropped it.")
    print("Evaluate by inspecting the highest-scoring readings instead:")
    print(df.nlargest(10, "anomaly_score")[
        ["ts", "location"] + FEATURES + ["anomaly_score"]].to_string(index=False))

## Cell 8: Plot the result

In [ ]:
import matplotlib.pyplot as plt

sub = df[df["location"] == "north-lab"].tail(2000)
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax[0].plot(sub["ts"], sub["temperature_c"], lw=0.8, label="temperature_c")
flagged = sub[sub["is_anomaly"]]
ax[0].scatter(flagged["ts"], flagged["temperature_c"],
              s=26, color="crimson", zorder=3, label="flagged")
ax[0].set_ylabel("°C"); ax[0].legend(loc="upper left")
ax[0].set_title("north-lab temperature with Random Cut Forest anomalies")

ax[1].plot(sub["ts"], sub["anomaly_score"], lw=0.8, color="#B8562B")
ax[1].axhline(cutoff, ls="--", color="grey", label=f"cutoff {cutoff:.2f}")
ax[1].set_ylabel("anomaly score"); ax[1].set_xlabel("epoch seconds")
ax[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

## Cell 9: Write the scored data to the curated prefix

In [ ]:
out = df[["ts", "device_id", "location"] + FEATURES
         + ["anomaly_score", "is_anomaly"]].copy()
buf = io.StringIO()
out.to_csv(buf, index=False)
key = "curated/scored/backfill-scored.csv"
s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue().encode())
print(f"Wrote s3://{BUCKET}/{key}  ({len(out):,} rows)")

## Cell 10: Record the cutoff for Lab 07

In [ ]:
print("ANOMALY_CUTOFF for Lab 07 =", round(float(cutoff), 4))